### ЗАДАЧА: Панель контроля рекламных бюджетов по паттерну `MVC`

Маркетинговая команда следит за рекламными кампаниями и проверяет кейсы с перерасходом.
Для каждого кейса нужно хранить:
- `case_id`
- `campaign`
- `channel`
- `planned_budget`
- `spent_budget`
- `overspend_amount` — вычисляемое поле `spent_budget - planned_budget`
- `status`
- `manager`
- `resolution`

Статусы кейса:
- `new`
- `reviewing`
- `approved`
- `frozen`

Бизнес-правила:
- нельзя создать кейс с уже существующим `case_id`;
- при создании нужно корректно считать `overspend_amount`;
- начать review можно только если назначен `manager`;
- одобрить кейс можно только из `reviewing`;
- заморозить кампанию можно только из `reviewing`;
- финальные кейсы (`approved`, `frozen`) нельзя менять дальше;
- при `approved` и `frozen` нужно записывать `resolution`.

In [ ]:
from dataclasses import dataclass

initial_cases = [
    ("PB-100", "spring_sale", "google_ads", 1000.0, 1450.0),
    ("PB-101", "brand_push", "yandex_ads", 800.0, 760.0),
]

actions = [
    ("show",),
    ("review", "PB-100"),
    ("assign", "PB-100", "Olga"),
    ("review", "PB-100"),
    ("approve", "PB-100", "budget_extended"),
    ("create", "PB-102", "new_arrivals", "tiktok_ads", 500.0, 950.0),
    ("assign", "PB-102", "Max"),
    ("freeze", "PB-102", "overspend_blocked"),
    ("show",),
]

@dataclass
class PromoBudgetCase:
    case_id: str
    campaign: str
    channel: str
    planned_budget: float
    spent_budget: float
    overspend_amount: float
    status: str = "new"
    manager: str | None = None
    resolution: str | None = None

class PromoBudgetModel:
    def __init__(self):
        # TODO 1: создайте self.cases как пустой словарь
        self.cases = {}

    def _calculate_overspend(self, planned_budget: float, spent_budget: float) -> float:
        # TODO 2: корректно вычислите перерасход (spent_budget - planned_budget)
        return spent_budget - planned_budget

    def add_case(self, case_id: str, campaign: str, channel: str, planned_budget: float, spent_budget: float) -> None:
        # TODO 3: если case_id уже существует, вызовите ValueError('Case already exists')
        if case_id in self.cases:
            raise ValueError("Case already exists")
        # TODO 4: вычислите overspend_amount через self._calculate_overspend
        overspend_amount = self._calculate_overspend(planned_budget, spent_budget)
        # TODO 5: создайте PromoBudgetCase и сохраните в self.cases
        self.cases[case_id] = PromoBudgetCase(
            case_id, campaign, channel, planned_budget, spent_budget, overspend_amount
        )

    def assign_manager(self, case_id: str, manager: str) -> None:
        # TODO 6: если кейс не найден, вызовите ValueError("Case not found")
        if case_id not in self.cases:
            raise ValueError("Case not found")
        # TODO 7: назначьте менеджера
        self.cases[case_id].manager = manager

    def start_review(self, case_id: str) -> None:
        # TODO 8: если кейс не найден, вызовите ValueError("Case not found")
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        # TODO 9: если менеджер не назначен, вызовите ValueError("Manager is required")
        if case.manager is None:
            raise ValueError("Manager is required")
        # TODO 10: если статус финальный (approved/frozen), вызовите ValueError("Final cases cannot be changed")
        if case.status in ("approved", "frozen"):
            raise ValueError("Final cases cannot be changed")
        # TODO 11: установите status = "reviewing"
        case.status = "reviewing"

    def approve_case(self, case_id: str, resolution: str) -> None:
        # TODO 12: если кейс не найден, вызовите ValueError("Case not found")
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        # TODO 13: если статус не "reviewing", вызовите ValueError("Can only approve from reviewing")
        if case.status != "reviewing":
            raise ValueError("Can only approve from reviewing")
        # TODO 14: установите status = "approved" и resolution
        case.status = "approved"
        case.resolution = resolution

    def freeze_case(self, case_id: str, resolution: str) -> None:
        # TODO 15: если кейс не найден, вызовите ValueError("Case not found")
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        # TODO 16: если статус не "reviewing", вызовите ValueError("Can only freeze from reviewing")
        if case.status != "reviewing":
            raise ValueError("Can only freeze from reviewing")
        # TODO 17: установите status = "frozen" и resolution
        case.status = "frozen"
        case.resolution = resolution


    def list_cases(self) -> list[str]:
        # TODO 18: верните список строк формата:
        #   'case_id | campaign | channel | planned_budget | spent_budget | overspend_amount | status | manager | resolution'
        rows = []
        for case in self.cases.values():
            manager = case.manager if case.manager else "None"
            resolution = case.resolution if case.resolution else "None"
            row = (
                f"{case.case_id} | {case.campaign} | {case.channel} | "
                f"{case.planned_budget:.2f} | {case.spent_budget:.2f} | "
                f"{case.overspend_amount:.2f} | {case.status} | {manager} | {resolution}"
            )
            rows.append(row)
        return rows

class PromoBudgetView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        # TODO 19: если rows пустой, выведите 'Кейсов нет'
        if not rows:
            print('Кейсов нет')
            return
        # TODO 20: иначе выведите заголовок 'Рекламные кейсы:' и все строки
        print('Рекламные кейсы:')
        for row in rows:
            print(row)

    @staticmethod
    def render_success(message: str) -> None:
        # TODO 21: выведите сообщение успеха
        print(f"Успех: {message}")

    @staticmethod
    def render_error(message: str) -> None:
        # TODO 22: выведите сообщение ошибки
        print(f"Ошибка: {message}")

class PromoBudgetController:
    def __init__(self, model: PromoBudgetModel, view: PromoBudgetView):
        # TODO 23: сохраните model и view
        self.model = model
        self.view = view

    def create_case(self, case_id: str, campaign: str, channel: str, planned_budget: float, spent_budget: float) -> None:
        # TODO 24: через try/except вызовите model.add_case(...)
        try:
            self.model.add_case(case_id, campaign, channel, planned_budget, spent_budget)
            # TODO 25: покажите success или error
            self.view.render_success(f"Кейс {case_id} создан")
        except ValueError as error:
            self.view.render_error(str(error))

    def assign_manager(self, case_id: str, manager: str) -> None:
        # TODO 26: через try/except вызовите model.assign_manager(...)
        try:
            self.model.assign_manager(case_id, manager)
            # TODO 27: покажите success или error
            self.view.render_success(f"Менеджер {manager} назначен на кейс {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def start_review(self, case_id: str) -> None:
        # TODO 28: через try/except вызовите model.start_review(...)
        try:
            self.model.start_review(case_id)
            # TODO 29: покажите success или error
            self.view.render_success(f"Кейс {case_id} отправлен на проверку")
        except ValueError as error:
            self.view.render_error(str(error))

    def approve_case(self, case_id: str, resolution: str) -> None:
        # TODO 30: через try/except вызовите model.approve_case(...)
        try:
            self.model.approve_case(case_id, resolution)
            # TODO 31: покажите success или error
            self.view.render_success(f"Кейс {case_id} одобрен с резолюцией '{resolution}'")
        except ValueError as error:
            self.view.render_error(str(error))

    def freeze_case(self, case_id: str, resolution: str) -> None:
        # TODO 32: через try/except вызовите model.freeze_case(...)
        try:
            self.model.freeze_case(case_id, resolution)
            # TODO 33: покажите success или error
            self.view.render_success(f"Кейс {case_id} заморожен с резолюцией '{resolution}'")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_cases(self) -> None:
        # TODO 34: получите строки из model.list_cases() и передайте их во view.render_cases(...)
        cases = self.model.list_cases()
        self.view.render_cases(cases)

model = PromoBudgetModel()
view = PromoBudgetView()
controller = PromoBudgetController(model, view)

# TODO 35: загрузите начальные кейсы из initial_cases в model
# Подсказка: пройдите по initial_cases циклом for
# и вызовите model.add_case(case_id, campaign, channel, planned_budget, spent_budget)
for case_id, campaign, channel, planned_budget, spent_budget in initial_cases:
    try:
        model.add_case(case_id, campaign, channel, planned_budget, spent_budget)
        view.render_success(f"Начальный кейс {case_id} загружен")
    except ValueError as e:
        view.render_error(f"Ошибка загрузки начального кейса {case_id}: {str(e)}")

# TODO 36: обработайте все действия из actions
for action in actions:
    # TODO 37: если action[0] == 'show', вызовите controller.show_cases()
    if action[0] == "show":
        controller.show_cases()
    # TODO 38: если action[0] == 'create', распакуйте case_id, campaign, channel, planned_budget, spent_budget
    # и вызовите controller.create_case(...)
    elif action[0] == "create":
        _, case_id, campaign, channel, planned_budget, spent_budget = action
        controller.create_case(case_id, campaign, channel, planned_budget, spent_budget)
    # TODO 39: если action[0] == 'assign', распакуйте case_id и manager
    # и вызовите controller.assign_manager(...)
    elif action[0] == "assign":
        _, case_id, manager = action
        controller.assign_manager(case_id, manager)
    # TODO 40: если action[0] == 'review', распакуйте case_id
    # и вызовите controller.start_review(...)
    elif action[0] == "review":
        _, case_id = action
        controller.start_review(case_id)
    # TODO 41: если action[0] == 'approve', распакуйте case_id и resolution
    # и вызовите controller.approve_case(...)
    elif action[0] == "approve":
        _, case_id, resolution = action
        controller.approve_case(case_id, resolution)
    # TODO 42: если action[0] == 'freeze', распакуйте case_id и resolution
    # и вызовите controller.freeze_case(...)
    elif action[0] == "freeze":
        _, case_id, resolution = action
        controller.freeze_case(case_id, resolution)

print()
print("Финальное состояние:")
# TODO 43: покажите финальное состояние через controller.show_cases()
controller.show_cases()


In [ ]:
# Promo budget cases:
# PB-100 | spring_sale | google_ads | 1000.00 | 1450.00 | 450.00 | new | None | None
# PB-101 | brand_push | yandex_ads | 800.00 | 760.00 | -40.00 | new | None | None
# ERROR: Manager is required
# SUCCESS: Manager assigned to PB-100
# SUCCESS: Case PB-100 moved to review
# SUCCESS: Case PB-100 approved
# SUCCESS: Case PB-102 created
# SUCCESS: Manager assigned to PB-102
# ERROR: Case is not in review
# Promo budget cases:
# PB-100 | spring_sale | google_ads | 1000.00 | 1450.00 | 450.00 | approved | Olga | budget_extended
# PB-101 | brand_push | yandex_ads | 800.00 | 760.00 | -40.00 | new | None | None
# PB-102 | new_arrivals | tiktok_ads | 500.00 | 950.00 | 450.00 | new | Max | None
#
# Финальное состояние:
# Promo budget cases:
# PB-100 | spring_sale | google_ads | 1000.00 | 1450.00 | 450.00 | approved | Olga | budget_extended
# PB-101 | brand_push | yandex_ads | 800.00 | 760.00 | -40.00 | new | None | None
# PB-102 | new_arrivals | tiktok_ads | 500.00 | 950.00 | 450.00 | new | Max | None